<a href="https://colab.research.google.com/github/ayushgkp/UCS547-Accelerated-Data-Science/blob/main/Assignment3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Q1.Write a CUDA C/C++ program to perform element-wise additon of two
vectors.
C[i]=A[i]+B[i]

In [1]:
%%writefile vecadd.cu
#include <stdio.h>
#define N 1024

__global__ void vecAdd(float *A, float *B, float *C) {
    int i = threadIdx.x + blockDim.x * blockIdx.x;
    if (i < N) C[i] = A[i] + B[i];
}

int main() {
    float *h_A, *h_B, *h_C;
    float *d_A, *d_B, *d_C;
    int size = N * sizeof(float);

    h_A = (float*)malloc(size);
    h_B = (float*)malloc(size);
    h_C = (float*)malloc(size);

    for (int i = 0; i < N; i++) { h_A[i] = i; h_B[i] = i * 2; }

    cudaMalloc(&d_A, size); cudaMalloc(&d_B, size); cudaMalloc(&d_C, size);
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    vecAdd<<<4, 256>>>(d_A, d_B, d_C);

    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);
    printf("C[0]=%.1f  C[1]=%.1f  C[1023]=%.1f\n", h_C[0], h_C[1], h_C[1023]);

    free(h_A); free(h_B); free(h_C);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    return 0;
}

Writing vecadd.cu


In [2]:
!nvcc vecadd.cu -o vecadd && ./vecadd

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
C[0]=0.0  C[1]=3.0  C[1023]=3069.0


Q2 — Thrust Vector Addition (N=1024)

In [3]:
%%writefile thrust_add.cu
#include <thrust/host_vector.h>
#include <thrust/device_vector.h>
#include <iostream>

int main() {
    int N = 1024;
    thrust::host_vector<float> h_A(N), h_B(N);
    for (int i = 0; i < N; i++) { h_A[i] = i; h_B[i] = i * 2; }

    thrust::device_vector<float> d_A = h_A, d_B = h_B;
    thrust::device_vector<float> d_C(N);

    thrust::transform(d_A.begin(), d_A.end(), d_B.begin(), d_C.begin(),
                      thrust::plus<float>());

    thrust::host_vector<float> h_C = d_C;
    std::cout << "C[0]=" << h_C[0] << "  C[1]=" << h_C[1]
              << "  C[1023]=" << h_C[1023] << std::endl;
    return 0;
}

Writing thrust_add.cu


In [4]:
!nvcc thrust_add.cu -o thrust_add && ./thrust_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
C[0]=0  C[1]=3  C[1023]=3069


Q3 — Thrust Dot Product vs CPU (N=1024)

In [5]:
%%writefile dot_product.cu
#include <thrust/host_vector.h>
#include <thrust/device_vector.h>
#include <thrust/inner_product.h>
#include <iostream>
#include <chrono>

int main() {
    int N = 1024;
    thrust::host_vector<float> h_A(N), h_B(N);
    for (int i = 0; i < N; i++) { h_A[i] = 1.0f; h_B[i] = 2.0f; }

    // CPU dot product
    auto t1 = std::chrono::high_resolution_clock::now();
    float cpu_result = 0;
    for (int i = 0; i < N; i++) cpu_result += h_A[i] * h_B[i];
    auto t2 = std::chrono::high_resolution_clock::now();
    double cpu_time = std::chrono::duration<double, std::micro>(t2 - t1).count();

    // GPU Thrust dot product
    thrust::device_vector<float> d_A = h_A, d_B = h_B;
    auto t3 = std::chrono::high_resolution_clock::now();
    float gpu_result = thrust::inner_product(d_A.begin(), d_A.end(), d_B.begin(), 0.0f);
    auto t4 = std::chrono::high_resolution_clock::now();
    double gpu_time = std::chrono::duration<double, std::micro>(t4 - t3).count();

    std::cout << "CPU Result: " << cpu_result << "  Time: " << cpu_time << " us\n";
    std::cout << "GPU Result: " << gpu_result << "  Time: " << gpu_time << " us\n";
    return 0;
}

Writing dot_product.cu


In [6]:
!nvcc dot_product.cu -o dot_product && ./dot_product

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
CPU Result: 2048  Time: 84.606 us
GPU Result: 2048  Time: 447881 us


Q4 — CUDA Matrix Multiplication (16×16) + Explanation

In [8]:
%%writefile matmul.cu
#include <stdio.h>
#define N 16

__global__ void matMul(float A[N][N], float B[N][N], float C[N][N]) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float sum = 0;
        for (int k = 0; k < N; k++) sum += A[row][k] * B[k][col];
        C[row][col] = sum;
    }
}

int main() {
    float h_A[N][N], h_B[N][N], h_C[N][N];
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++) { h_A[i][j] = 1.0f; h_B[i][j] = 2.0f; }

    float (*d_A)[N], (*d_B)[N], (*d_C)[N];
    cudaMalloc(&d_A, N*N*sizeof(float));
    cudaMalloc(&d_B, N*N*sizeof(float));
    cudaMalloc(&d_C, N*N*sizeof(float));
    cudaMemcpy(d_A, h_A, N*N*sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, N*N*sizeof(float), cudaMemcpyHostToDevice);

    dim3 threads(16, 16), blocks(1, 1);
    matMul<<<blocks, threads>>>(d_A, d_B, d_C);

    cudaMemcpy(h_C, d_C, N*N*sizeof(float), cudaMemcpyDeviceToHost);
    printf("C[0][0]=%.1f  (expected 32.0 for 16x1 dot 1x2)\n", h_C[0][0]);

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    return 0;
}

Overwriting matmul.cu


In [9]:
!nvcc matmul.cu -o matmul && ./matmul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
C[0][0]=32.0  (expected 32.0 for 16x1 dot 1x2)


Why MatMul needs more computation than addition (write as a comment or text cell):

Vector addition does N multiplications — one per element, O(N).
Matrix multiplication C=A×B for N×N matrices does N³ multiplications — each of the N² output elements requires N multiply-adds. For N=16: addition = 16 ops, matmul = 4096 ops. That's why a parallel GPU kernel with 2D thread blocks is critical for matrix multiplication.

Q5 — Vector Addition Size 5,000,000: CPU vs CUDA vs Thrust vs RAPIDS

In [10]:
%%writefile compare_all.cu
#include <thrust/host_vector.h>
#include <thrust/device_vector.h>
#include <stdio.h>
#include <chrono>
#define N 5000000

// CUDA kernel
__global__ void vecAddKernel(float *A, float *B, float *C, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) C[i] = A[i] + B[i];
}

int main() {
    // --- CPU ---
    float *h_A = new float[N], *h_B = new float[N], *h_C = new float[N];
    for (int i = 0; i < N; i++) { h_A[i] = 1.0f; h_B[i] = 2.0f; }

    auto t1 = std::chrono::high_resolution_clock::now();
    for (int i = 0; i < N; i++) h_C[i] = h_A[i] + h_B[i];
    auto t2 = std::chrono::high_resolution_clock::now();
    double cpu_ms = std::chrono::duration<double, std::milli>(t2 - t1).count();

    // --- CUDA ---
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, N*sizeof(float));
    cudaMalloc(&d_B, N*sizeof(float));
    cudaMalloc(&d_C, N*sizeof(float));
    cudaMemcpy(d_A, h_A, N*sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, N*sizeof(float), cudaMemcpyHostToDevice);

    cudaEvent_t start, stop; float cuda_ms;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    cudaEventRecord(start);
    vecAddKernel<<<(N+255)/256, 256>>>(d_A, d_B, d_C, N);
    cudaEventRecord(stop); cudaEventSynchronize(stop);
    cudaEventElapsedTime(&cuda_ms, start, stop);

    // --- Thrust ---
    thrust::device_vector<float> td_A(h_A, h_A+N), td_B(h_B, h_B+N), td_C(N);
    cudaEventRecord(start);
    thrust::transform(td_A.begin(), td_A.end(), td_B.begin(), td_C.begin(),
                      thrust::plus<float>());
    cudaEventRecord(stop); cudaEventSynchronize(stop);
    float thrust_ms;
    cudaEventElapsedTime(&thrust_ms, start, stop);

    printf("\n=== Performance Comparison (N = 5,000,000) ===\n");
    printf("%-20s %-15s %-20s\n", "Method", "Time (ms)", "Notes");
    printf("%-20s %-15.3f %-20s\n", "CPU (sequential)", cpu_ms, "Single thread");
    printf("%-20s %-15.3f %-20s\n", "CUDA kernel", cuda_ms, "Parallel GPU");
    printf("%-20s %-15.3f %-20s\n", "Thrust", thrust_ms, "GPU abstraction");
    printf("RAPIDS: run separately in Python (see below)\n");

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    delete[] h_A; delete[] h_B; delete[] h_C;
    return 0;
}

Writing compare_all.cu


In [11]:
!nvcc compare_all.cu -o compare_all && ./compare_all

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).

=== Performance Comparison (N = 5,000,000) ===
Method               Time (ms)       Notes               
CPU (sequential)     21.913          Single thread       
CUDA kernel          18.713          Parallel GPU        
Thrust               0.254           GPU abstraction     
RAPIDS: run separately in Python (see below)


In [12]:
import cupy as cp
import time

N = 5_000_000
a = cp.ones(N, dtype=cp.float32)
b = cp.full(N, 2.0, dtype=cp.float32)

cp.cuda.Stream.null.synchronize()
t1 = time.perf_counter()
c = a + b
cp.cuda.Stream.null.synchronize()
t2 = time.perf_counter()

print(f"RAPIDS (CuPy) Time: {(t2-t1)*1000:.3f} ms")
print(f"Result check c[0] = {c[0]}")

RAPIDS (CuPy) Time: 106.629 ms
Result check c[0] = 3.0


Q6 — Thrust Sum of Vector (size 10, values 1–10)

In [14]:
%%writefile thrust_sum.cu
#include <thrust/device_vector.h>
#include <thrust/sequence.h>
#include <thrust/reduce.h>
#include <iostream>

int main() {
    thrust::device_vector<int> d_vec(10);
    thrust::sequence(d_vec.begin(), d_vec.end(), 1);  // fills 1,2,...,10

    int sum = thrust::reduce(d_vec.begin(), d_vec.end(), 0, thrust::plus<int>());
    std::cout << "Sum of elements 1 to 10 = " << sum << std::endl;
    return 0;
}

Overwriting thrust_sum.cu


In [15]:
!nvcc thrust_sum.cu -o thrust_sum && ./thrust_sum

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Sum of elements 1 to 10 = 55


Q7 — Thrust Sort Ascending (values: 7,2,9,1,5,3,8,4)

In [17]:
%%writefile thrust_sort.cu
#include <thrust/device_vector.h>
#include <thrust/sort.h>
#include <iostream>

int main() {
    int data[] = {7, 2, 9, 1, 5, 3, 8, 4};
    int n = 8;

    thrust::device_vector<int> d_vec(data, data + n);

    std::cout << "Before sorting: ";
    for (int i = 0; i < n; i++) std::cout << (int)d_vec[i] << " ";
    std::cout << std::endl;

    thrust::sort(d_vec.begin(), d_vec.end());

    std::cout << "After sorting:  ";
    for (int i = 0; i < n; i++) std::cout << (int)d_vec[i] << " ";
    std::cout << std::endl;

    return 0;
}

Overwriting thrust_sort.cu


In [18]:
!nvcc thrust_sort.cu -o thrust_sort && ./thrust_sort


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Before sorting: 7 2 9 1 5 3 8 4 
After sorting:  1 2 3 4 5 7 8 9 
